## 1️⃣ Clone do Repositório

In [ ]:
import os, sys

REPO_URL = "https://github.com/felipe-huszar/medical-llm-assistant.git"
REPO_DIR = "/content/medical-llm-assistant"

if os.path.exists(REPO_DIR):
    print("Repositório já existe — atualizando...")
    !cd {REPO_DIR} && git pull
    # Limpa cache de módulos src/ para pegar código novo
    removed = [sys.modules.pop(k) for k in list(sys.modules) if k.startswith("src.")]
    if removed:
        print(f"{len(removed)} módulos src/ recarregados do cache")
else:
    print("Clonando repositório...")
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
print("Repositorio pronto.")
!ls -la


## 2️⃣ Instalação de Dependências

In [ ]:
import importlib.util

# Verifica chromadb (instalado junto com o resto — nunca vem pré-instalado no Colab)
if importlib.util.find_spec('chromadb') is None:
    print('📦 Instalando dependências...')
    import subprocess
    subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'])
    subprocess.run(['pip', 'install', '-q', 'unsloth', 'peft', 'bitsandbytes', 'accelerate', 'gdown'])
    print('🔄 Reiniciando runtime... Execute esta célula novamente após o restart.')
    import os; os.kill(os.getpid(), 9)
else:
    print('✅ Dependências instaladas.')


## 3️⃣ Configuração de Ambiente

In [ ]:
# Google Drive — montar para acessar modelo treinado
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive montado.')


In [ ]:
import os, sys

# ──────────────────────────────────────────────────────────
# ⬇️  MUDE AQUI para alternar entre MockLLM e modelo real
# ──────────────────────────────────────────────────────────
USE_MOCK = 'false'   # 'false' para usar modelo real (requer GPU)

# ID da pasta pública do Drive com o modelo
# https://drive.google.com/drive/folders/1KQcHlFcoHx3VpqN6vN3eYRHiCjt6p-1Z
DRIVE_FOLDER_ID = '1KQcHlFcoHx3VpqN6vN3eYRHiCjt6p-1Z'
LORA_LOCAL_PATH = '/content/model'

os.environ['USE_MOCK_LLM']  = USE_MOCK
os.environ['MODEL_PATH']    = LORA_LOCAL_PATH
os.environ['BASE_MODEL_ID'] = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'
os.environ['CHROMA_DB_PATH']= '/content/chroma_db'

sys.path.insert(0, '/content/medical-llm-assistant')

print('=' * 55)
print('  🔧  CONFIGURAÇÃO ATUAL')
print('=' * 55)
print(f"  USE_MOCK_LLM   = {os.environ['USE_MOCK_LLM']}")
print(f"  MODEL_PATH     = {os.environ['MODEL_PATH']}")
print(f"  BASE_MODEL_ID  = {os.environ['BASE_MODEL_ID']}")
print('=' * 55)

# Download do modelo quando modo real
if USE_MOCK.lower() != 'true':
    import subprocess
    import zipfile
    import glob
    
    print('\n📥 Baixando modelo do Drive (pasta pública)...')
    
    # Cria diretório
    os.makedirs(LORA_LOCAL_PATH, exist_ok=True)
    
    # Download via gdown (pasta pública)
    result = subprocess.run(
        ['gdown', '--folder', f'https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}',
         '-O', LORA_LOCAL_PATH, '--remaining-ok'],
        capture_output=True, text=True
    )
    
    if result.returncode != 0:
        print('⚠️  Erro no gdown:', result.stderr)
        print('\n🔧 Tentando método alternativo (mount Drive)...')
        # Fallback: monta Drive
        from google.colab import drive
        drive.mount('/content/drive')
        import shutil
        shutil.copy(f'/content/drive/MyDrive/FIAP-TC-Fase3/medical_assistant_llm.zip', 
                    f'{LORA_LOCAL_PATH}/medical_assistant_llm.zip')
    else:
        print('✅ Download concluído.')
    
    # Procura e extrai ZIP
    zips = glob.glob(f'{LORA_LOCAL_PATH}/*.zip')
    if zips:
        zip_path = zips[0]
        print(f'📦 Extraindo: {os.path.basename(zip_path)}')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(LORA_LOCAL_PATH)
        print('✅ Modelo extraído.')
    else:
        print('⚠️  Nenhum ZIP encontrado. Verificando arquivos...')
    
    # Valida
    cfg = None
    wts = None
    for root, _, files in os.walk(LORA_LOCAL_PATH):
        if 'adapter_config.json' in files and 'adapter_model.safetensors' in files:
            cfg = os.path.join(root, 'adapter_config.json')
            wts = os.path.join(root, 'adapter_model.safetensors')
            break
    
    if cfg and wts:
        final_model_path = os.path.dirname(cfg)
        os.environ['MODEL_PATH'] = final_model_path
        print('✅ Modelo validado!')
        print(f'📁 MODEL_PATH = {final_model_path}')
    else:
        print('❌ Erro: adapter_config.json ou adapter_model.safetensors não encontrados')
        print(f'   Conteúdo de {LORA_LOCAL_PATH}:')
        !ls -la {LORA_LOCAL_PATH}
else:
    print('\nℹ️  Modo Mock ativo — modelo não necessário.')


## ⚠️ Limpeza do ChromaDB

> **ATENÇÃO:** Isso apaga **todos** os pacientes e históricos. Execute apenas se tiver certeza.


In [ ]:
# ⚠️ LIMPEZA TOTAL DO CHROMADB — execute apenas se tiver certeza!
CONFIRMAR = False  # mude para True para confirmar

if CONFIRMAR:
    import src.rag.patient_rag as rag
    # Limpa via API (sem restart necessário)
    client = rag._get_client()
    for col_name in ['patients', 'consultations']:
        try:
            client.delete_collection(col_name)
            print(f'✅ Collection {col_name!r} removida')
        except Exception:
            print(f'ℹ️  Collection {col_name!r} não existia')
    # Reseta o cache do cliente para forçar recriação
    rag._client = None
    rag._client_path = None
    # Re-seed
    from src.rag.patient_rag import seed_from_file
    n = seed_from_file('data/patients_seed.json')
    print(f'✅ {n} pacientes seed reinseridos. ChromaDB limpo e pronto!')
else:
    print('⚠️  Limpeza NÃO executada. Mude CONFIRMAR = True para prosseguir.')


In [ ]:
# Carrega o modelo UMA vez e guarda no cache da factory
# A celula de Gradio reutiliza esse cache sem recarregar
import os, src.llm.factory as _factory

if os.environ.get('USE_MOCK_LLM', 'true').lower() != 'true':
    if _factory._cached_llm is None:
        print('Carregando modelo na memoria (unica vez)...')
        _factory.get_llm()  # popula _cached_llm internamente
        print('Modelo pronto e em cache.')
    else:
        print('Modelo ja estava em cache, sem recarga.')
else:
    print('Modo Mock ativo, modelo nao necessario.')


## 3️⃣ Interface Gradio (UI Completa)

Clique no link público gerado para abrir a interface.

**Abas disponíveis:**
- `👤 Paciente` — Busca/cadastro por CPF
- `🩺 Consulta` — Pergunta clínica + resposta da IA

In [ ]:
import sys

# Remove app e src/ do cache — EXCETO src.llm.* que contem o modelo em VRAM
# Apagar src.llm.factory limparia _cached_llm e forcaria reload do modelo
for mod in list(sys.modules):
    if mod == "app" or (mod.startswith("src.") and not mod.startswith("src.llm")):
        del sys.modules[mod]

import app as _app
import src.llm.factory as _factory

# Reutiliza modelo ja em VRAM (cache em src.llm.factory._cached_llm)
if _factory._cached_llm is not None:
    _app.set_llm(_factory._cached_llm)
    print("Modelo reutilizado do cache.")
else:
    print("⚠️ Modelo não está em cache. Rode a célula anterior primeiro.")

_app.demo.queue().queue().launch(share=True)


## 4️⃣ Execução dos Testes

In [ ]:
# ── Testes Unitários (Safety Gate)
print('🧪 Testes Unitários — Safety Gate')
print('-' * 55)
!python -m pytest tests/unit -v --tb=short

In [ ]:
# ── Testes de Integração (Pipeline LangGraph)
print('🧪 Testes de Integração — Pipeline')
print('-' * 55)
!python -m pytest tests/integration -v --tb=short

In [ ]:
# ── Testes E2E (Jornadas Completas + Casos Extendidos)
print('🧪 Testes E2E — Jornadas Completas')
print('-' * 55)
!python -m pytest tests/e2e -v --tb=short

In [ ]:
# ── Sumário consolidado de todos os testes
print('📊 SUMÁRIO COMPLETO')
print('-' * 55)
!python -m pytest tests/ --tb=no -q